# 手撕DeepSeek MLA

不考虑效率

仅复现原论文公式

> ref: DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts Language Model

# setting

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
# @dataclass
class ModelArgs:
    dim: int = 64
    n_heads: int = 8
    n_kv_heads: int =  2

    # down 两者远小于 dim
    dc_kv: int = 4 
    dc_q: int = 4

bs = 3
seq_len = 5

config = ModelArgs()
print(config)

<__main__.ModelArgs object at 0x11a167e10>

In [3]:
h = torch.randn(bs, seq_len, config.dim)

## Standard Multi-Heads Attention

参考Llama3-GQA， 去除rope简易实现

In [4]:
# ref : Llama3 GQA, ./notebook/Llama3-GQA.ipynb
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """torch.repeat_interleave(x, dim=2, repeats=n_rep)"""
    bs, slen, n_kv_heads, head_dim = x.shape
    if n_rep == 1:
        return x
    return (
        x[:, :, :, None, :]
        .expand(bs, slen, n_kv_heads, n_rep, head_dim) # 
        .reshape(bs, slen, n_kv_heads * n_rep, head_dim)
    )

class MultiHeadsAttention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.n_kv_heads = args.n_heads if args.n_kv_heads is None else args.n_kv_heads
        self.head_dim = args.dim // args.n_heads # 18/6 = 3
        self.n_rep = self.n_heads // self.n_kv_heads

        self.wq = nn.Linear(in_features=args.dim, out_features=args.n_heads * self.head_dim,bias=False,)
        self.wk = nn.Linear(in_features=args.dim, out_features=args.n_kv_heads * self.head_dim,bias=False,)
        self.wv = nn.Linear(in_features=args.dim, out_features=args.n_kv_heads * self.head_dim,bias=False,)
        self.wo = nn.Linear(in_features=args.n_heads * self.head_dim, out_features=args.dim,bias=False,)
        
    def forward(
        self,
        x: torch.Tensor,
    ):
        bsz, seqlen, _ = x.shape
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)

        # here we ignore RoPE

        xq = xq.view(bsz, seqlen, self.n_heads, self.head_dim)
        xk = xk.view(bsz, seqlen, self.n_kv_heads, self.head_dim)
        xv = xv.view(bsz, seqlen, self.n_kv_heads, self.head_dim)
        
        keys = repeat_kv( xk, self.n_rep )  
        values = repeat_kv( xv, self.n_rep )  
        
        xq = xq.transpose(1, 2)  # (bs, n_local_heads, seqlen, head_dim)
        keys = keys.transpose(1, 2)  # (bs, n_local_heads, seqlen, head_dim)
        values = values.transpose(1, 2)  # (bs, n_local_heads, seqlen, head_dim)

        
        scores = xq @ keys.transpose(2, 3) / math.sqrt(self.head_dim)
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)
        output = scores @ values # (bs, n_local_heads, seqlen, head_dim)
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)
        mha_output = self.wo(output)
        return mha_output

In [5]:
mha = MultiHeadsAttention(config)
print(mha)

MultiHeadsAttention(
  (wq): Linear(in_features=64, out_features=64, bias=False)
  (wk): Linear(in_features=64, out_features=16, bias=False)
  (wv): Linear(in_features=64, out_features=16, bias=False)
  (wo): Linear(in_features=64, out_features=64, bias=False)
)

In [6]:
out = mha(h)
print(h.shape)
print(out.shape)

torch.Size([3, 5, 64])

torch.Size([3, 5, 64])

# Multi-Heads Latent Attention

## model

1. 以下用down和up权重矩阵，代替直接的wq,wk,wv
2. MLA矩阵发生于训练之时

In [7]:
class MultiHeadsLatentAttention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.dim = args.dim
        # self.n_kv_heads = args.n_heads if args.n_kv_heads is None else args.n_kv_heads
        self.head_dim = args.dim // args.n_heads # 18/6 = 3
        self.dc_kv = args.dc_kv
        self.dc_q = args.dc_q

        # MLA Structure
        self.wq_down = nn.Linear(in_features=args.dim, out_features=args.dc_q, bias=False,)
        self.wq_up = nn.Linear(in_features=args.dc_q, out_features=args.dim , bias=False,)

        self.wkv_down = nn.Linear(in_features=args.dim, out_features=args.dc_kv, bias=False,)
        self.wk_up = nn.Linear(in_features=args.dc_kv, out_features=args.dim, bias=False,)
        self.wv_up = nn.Linear(in_features=args.dc_kv, out_features=args.dim, bias=False,)
        
        self.wo = nn.Linear(in_features=args.dim, out_features=args.dim,bias=False,)


In [8]:
mla = MultiHeadsLatentAttention(config)
print(mla)

MultiHeadsLatentAttention(
  (wq_down): Linear(in_features=64, out_features=4, bias=False)
  (wq_up): Linear(in_features=4, out_features=64, bias=False)
  (wkv_down): Linear(in_features=64, out_features=4, bias=False)
  (wk_up): Linear(in_features=4, out_features=64, bias=False)
  (wv_up): Linear(in_features=4, out_features=64, bias=False)
  (wo): Linear(in_features=64, out_features=64, bias=False)
)

## forward

In [9]:
# 该段代码是关键，先降维，再升维
# xq = mha.wq(h)
c_q = mla.wq_down(h)
xq = mla.wq_up(c_q)

# xk = mha.wk(h)
# xk = mha.wv(h)
c_kv = mla.wkv_down(h)
xk = mla.wk_up(c_kv)
xv = mla.wv_up(c_kv)

print(xq.shape)
print(xk.shape)
print(xv.shape)

torch.Size([3, 5, 64])

torch.Size([3, 5, 64])

torch.Size([3, 5, 64])

In [10]:
# 与传统多头注意力无差别
xq = xq.view(bs, seq_len, mla.n_heads, mla.head_dim)
xk = xk.view(bs, seq_len, mla.n_heads, mla.head_dim)
xv = xv.view(bs, seq_len, mla.n_heads, mla.head_dim)

query = xq.transpose(1,2)
key = xk.transpose(1,2) # bs, n_heads, [seq_len, head_dim]
value = xv.transpose(1,2)
print(query.shape) 
print(key.shape)

scores = query @ key.transpose(2,3) # keys: bs, n_heads, [head_dim, seq_len]
scores = F.softmax(scores.float(), dim=-1).type_as(xq)
output = scores @ value 
output = output.transpose(1, 2).contiguous().view(bs, seq_len, -1)
output = mla.wo(output)
print(output)

torch.Size([3, 8, 5, 8])

torch.Size([3, 8, 5, 8])

tensor([[[ 0.0810, -0.0963, -0.0708, -0.0033,  0.0213,  0.0988,  0.1013,
          -0.0331, -0.0506, -0.1009,  0.0875,  0.1021,  0.0129, -0.0416,
           0.1533, -0.0653, -0.1009, -0.0826, -0.1331, -0.1047,  0.0412,
           0.0380, -0.0164,  0.0919, -0.0666,  0.0324, -0.0445, -0.0544,
          -0.1633, -0.0429,  0.0184,  0.0115, -0.0198, -0.1259,  0.0008,
          -0.0982,  0.0307, -0.0700, -0.0476,  0.0458,  0.0442,  0.0267,
          -0.1350, -0.0524, -0.0849,  0.0720,  0.0585, -0.0457, -0.0842,
           0.1186,  0.0749, -0.0992,  0.0425,  0.0130,  0.0603,  0.0678,
          -0.0677,  0.0291, -0.0333, -0.0011,  0.1253, -0.1358, -0.1240,
           0.0708],
         [ 0.0593, -0.1185, -0.0620, -0.0026,  0.0306,  0.1117,  0.0596,
          -0.0577, -0.0027, -0.0138,  0.0708,  0.1187,  0.0213, -0.0845,
           0.1643, -0.0295, -0.0279, -0.0427, -0.0624, -0.0782,  0.0283,
           0.0665, -0.0014,  0.0849, -0.0390,  0.0451, -0.0349, -0.0797,
          -0.1581, -0.0435,  0.0569, -0.0012, -0.0358, -0.0674,  0.0267,
          -0.0875,  0.0523, -0.0076, -0.0178,  0.0262,  0.0209,  0.0149,
          -0.1220, -0.0780, -0.0468,  0.0846,  0.0247, -0.0528, -0.0721,
           0.1321,  0.0215, -0.1216,  0.0293, -0.0058,  0.0609,  0.1054,
          -0.0333,  0.0322, -0.0577, -0.0232,  0.1525, -0.0998, -0.0857,
           0.0163],
         [ 0.0779, -0.1000, -0.0792,  0.0150,  0.0162,  0.0998,  0.0845,
          -0.0296, -0.0278, -0.0879,  0.0736,  0.1158,  0.0116, -0.0584,
           0.1806, -0.0500, -0.0935, -0.0882, -0.1124, -0.0824,  0.0321,
           0.0454, -0.0047,  0.0776, -0.0537,  0.0207, -0.0401, -0.0646,
          -0.1437, -0.0441,  0.0303,  0.0050, -0.0255, -0.1068,  0.0057,
          -0.0872,  0.0392, -0.0592, -0.0589,  0.0450,  0.0193,  0.0101,
          -0.1411, -0.0670, -0.0711,  0.0918,  0.0420, -0.0528, -0.0799,
           0.1455,  0.0457, -0.1041,  0.0395,  0.0055,  0.0447,  0.0865,
          -0.0524,  0.0245, -0.0410,  0.0029,  0.1322, -0.1428, -0.1108,
           0.0542],
         [ 0.0617, -0.1170, -0.0559, -0.0179,  0.0302,  0.1056,  0.0493,
          -0.0712, -0.0050, -0.0028,  0.0720,  0.1007,  0.0291, -0.0937,
           0.1550, -0.0210, -0.0257, -0.0270, -0.0620, -0.0792,  0.0304,
           0.0718, -0.0120,  0.0875, -0.0438,  0.0630, -0.0380, -0.0691,
          -0.1650, -0.0512,  0.0485,  0.0082, -0.0342, -0.0708,  0.0250,
          -0.0817,  0.0523,  0.0032, -0.0137,  0.0206,  0.0204,  0.0180,
          -0.1249, -0.0781, -0.0418,  0.0832,  0.0305, -0.0425, -0.0737,
           0.1163,  0.0237, -0.1153,  0.0263, -0.0053,  0.0621,  0.1019,
          -0.0293,  0.0307, -0.0562, -0.0195,  0.1474, -0.0880, -0.0772,
           0.0082],
         [ 0.0657, -0.1125, -0.0537, -0.0279,  0.0326,  0.1044,  0.0600,
          -0.0717, -0.0177, -0.0197,  0.0789,  0.0917,  0.0299, -0.0813,
           0.1444, -0.0280, -0.0391, -0.0284, -0.0790, -0.0912,  0.0344,
           0.0655, -0.0174,  0.0921, -0.0515,  0.0683, -0.0394, -0.0604,
          -0.1741, -0.0499,  0.0389,  0.0127, -0.0310, -0.0873,  0.0196,
          -0.0841,  0.0447, -0.0076, -0.0122,  0.0197,  0.0344,  0.0249,
          -0.1234, -0.0694, -0.0504,  0.0730,  0.0421, -0.0376, -0.0770,
           0.1034,  0.0414, -0.1106,  0.0287, -0.0016,  0.0681,  0.0891,
          -0.0383,  0.0320, -0.0512, -0.0172,  0.1399, -0.0893, -0.0837,
           0.0190]],

        [[-0.0234, -0.0340, -0.0161,  0.1022, -0.0466,  0.0099,  0.0365,
          -0.0592,  0.0459, -0.1287, -0.0048,  0.0009, -0.0100,  0.0553,
           0.0657, -0.0414, -0.1380,  0.0349,  0.0625,  0.0067,  0.0214,
           0.1444, -0.0308,  0.0347, -0.0758,  0.0025, -0.0237,  0.0173,
          -0.1265,  0.0247,  0.0461,  0.1054, -0.1916, -0.1200, -0.0017,
          -0.0095, -0.2505, -0.0857, -0.0657,  0.0070, -0.0730, -0.0253,
          -0.0939, -0.0924, -0.0197,  0.0670,  0.0302,  0.1067, -0.0678,
           0.0542, -0.0585,  0.0289,  0.0673,  0.0595, -0.0013,  0.1056,
           0.0626,  0.2027, -

## 矩阵吸收

以上参数发生在训练之时，训练完成后，我们可以做吸收操作，具体指wq参数和Wo参数，

我们写出如下等式：

Q = wq_up * ( wq_down * h ) = (wq_up * wq_down) * h

Q = wq * h 

目的是什么？

1. 训练时省显存
2. 训练完，推理Q满矩阵保精度，
3. 由于KV Cache的存在，decoding阶段时one-by-one token进行q计算
4. KV Cache极具减少存储体现。

In [11]:
wq = mla.wq_up.weight.data @ mla.wq_down.weight.data 

### 矩阵吸收后的forward(非训练阶段）

In [12]:
# 该段代码是关键，先降维，再升维
xq =  h @ wq
# c_q = mla.wq_down(h) #去除
# xq = mla.wq_up(c_q) #去除

c_kv = mla.wkv_down(h)
xk = mla.wk_up(c_kv)
# xv = mla.wv_up(c_kv) # 去除

# 与传统多头注意力无差别
xq = xq.view(bs, seq_len, mla.n_heads, mla.head_dim)
xk = xk.view(bs, seq_len, mla.n_heads, mla.head_dim)
xv = xv.view(bs, seq_len, mla.n_heads, mla.head_dim)

query = xq.transpose(1,2)
key = xk.transpose(1,2) # bs, n_heads, [seq_len, head_dim]
value = xv.transpose(1,2)
print(query.shape) 
print(key.shape)

scores = query @ key.transpose(2,3) # keys: bs, n_heads, [head_dim, seq_len]
scores = F.softmax(scores.float(), dim=-1).type_as(xq)
output = scores @ value 
output = output.transpose(1, 2).contiguous().view(bs, seq_len, -1)
output = mla.wo(output)
print(output)

torch.Size([3, 8, 5, 8])

torch.Size([3, 8, 5, 8])

tensor([[[ 0.0368, -0.1027,  0.0082,  0.0516, -0.0009,  0.1197,  0.0977,
          -0.0496, -0.0171, -0.0152,  0.0725,  0.1204, -0.0585, -0.0442,
           0.1138, -0.0762,  0.0118, -0.0474, -0.0585, -0.0970,  0.0505,
           0.0881,  0.0050,  0.1130, -0.0349,  0.0108, -0.0603, -0.1154,
          -0.1526, -0.0762,  0.0739, -0.0340, -0.0463, -0.0385,  0.0275,
          -0.1551,  0.0708, -0.0159, -0.0581,  0.0266,  0.0078,  0.0160,
          -0.0811, -0.1025, -0.0468,  0.0836, -0.0106, -0.0441, -0.0688,
           0.1449,  0.0027, -0.1500,  0.0356,  0.0019,  0.0609,  0.1393,
          -0.0358,  0.0228, -0.0253, -0.0376,  0.1715, -0.1119, -0.1472,
           0.0257],
         [ 0.0411, -0.0981,  0.0141,  0.0657, -0.0057,  0.1226,  0.0898,
          -0.0451, -0.0223, -0.0168,  0.0675,  0.1256, -0.0654, -0.0488,
           0.1188, -0.0839,  0.0159, -0.0577, -0.0599, -0.0893,  0.0554,
           0.0929,  0.0115,  0.1063, -0.0429,  0.0030, -0.0645, -0.1159,
          -0.1477, -0.0850,  0.0740, -0.0393, -0.0497, -0.0319,  0.0263,
          -0.1612,  0.0744, -0.0132, -0.0734,  0.0325,  0.0012,  0.0083,
          -0.0787, -0.1084, -0.0483,  0.0959, -0.0199, -0.0408, -0.0749,
           0.1556,  0.0017, -0.1536,  0.0339,  0.0023,  0.0519,  0.1596,
          -0.0379,  0.0200, -0.0231, -0.0272,  0.1752, -0.1211, -0.1476,
           0.0302],
         [ 0.1151, -0.1244, -0.1148, -0.0392,  0.0262,  0.0933, -0.0133,
          -0.0850, -0.0154, -0.0515,  0.0752,  0.0521,  0.1105, -0.1416,
           0.1930,  0.0021, -0.1349, -0.0559, -0.1125, -0.0566,  0.0301,
           0.0661, -0.0493,  0.0495, -0.0725,  0.0792, -0.0516,  0.0111,
          -0.1707, -0.0531, -0.0016,  0.0811, -0.0367, -0.1380,  0.0097,
          -0.0164,  0.0217, -0.0604, -0.0290,  0.0697, -0.0049,  0.0011,
          -0.2023, -0.0786, -0.0469,  0.0911,  0.0677, -0.0459, -0.0997,
           0.0885,  0.0632, -0.0732,  0.0384,  0.0283,  0.0310,  0.0612,
          -0.0541,  0.0564, -0.0830,  0.0319,  0.1408, -0.0855, -0.0452,
           0.0372],
         [ 0.0825, -0.1110, -0.0963, -0.0729,  0.0361,  0.0688,  0.0742,
          -0.0523, -0.0212, -0.0432,  0.0906,  0.0937,  0.0693, -0.1114,
           0.1713, -0.0074, -0.1031, -0.0444, -0.0936, -0.0805,  0.0021,
           0.0332, -0.0339,  0.0808, -0.0516,  0.0898, -0.0260, -0.0614,
          -0.1531, -0.0406,  0.0080,  0.0451, -0.0139, -0.1110,  0.0188,
          -0.0313,  0.0301, -0.0199, -0.0033,  0.0385,  0.0233,  0.0266,
          -0.1652, -0.0535, -0.0576,  0.0808,  0.0737, -0.0500, -0.0704,
           0.0981,  0.0361, -0.0768,  0.0340,  0.0149,  0.0731,  0.0565,
          -0.0236,  0.0118, -0.0575, -0.0071,  0.1162, -0.1108, -0.0631,
           0.0093],
         [ 0.0355, -0.1112, -0.0493, -0.0127,  0.0270,  0.0986,  0.1156,
          -0.0483, -0.0122, -0.0081,  0.0843,  0.1198, -0.0116, -0.0436,
           0.1253, -0.0430,  0.0007, -0.0209, -0.0644, -0.1104,  0.0343,
           0.0573, -0.0173,  0.1301, -0.0218,  0.0465, -0.0303, -0.1045,
          -0.1616, -0.0365,  0.0614, -0.0180, -0.0092, -0.0658,  0.0222,
          -0.1222,  0.0663, -0.0019,  0.0069,  0.0063,  0.0477,  0.0507,
          -0.1052, -0.0575, -0.0580,  0.0580,  0.0359, -0.0514, -0.0477,
           0.1144,  0.0257, -0.1193,  0.0301, -0.0134,  0.0846,  0.0687,
          -0.0308,  0.0187, -0.0345, -0.0589,  0.1339, -0.0927, -0.1276,
           0.0168]],

        [[-0.0695, -0.0351,  0.0167,  0.0852, -0.0457, -0.0332,  0.0857,
          -0.0640,  0.0430, -0.1213,  0.0216,  0.0062, -0.0251,  0.1029,
          -0.0013, -0.0475, -0.1242,  0.0625,  0.0231, -0.0100,  0.0227,
           0.1334, -0.0498,  0.0960, -0.0560, -0.0117, -0.0201, -0.0311,
          -0.1018,  0.0012,  0.0542,  0.1049, -0.1694, -0.1313, -0.0536,
          -0.0131, -0.2442, -0.1367, -0.0774,  0.0044, -0.0491, -0.0101,
          -0.1198, -0.0459, -0.0219,  0.0279,  0.0285,  0.1121, -0.0324,
           0.0087, -0.0599,  0.0167,  0.0567,  0.1258,  0.0146,  0.0506,
           0.0861,  0.1949,  

In [13]:
import torch

# 假设 tensor 是形状为 (a, b, c) 的张量
tensor = torch.randn(2, 3, 4)

# 扩展第 1 维，复制 8 份
expanded_tensor = tensor.repeat(8, 1, 1)
print(expanded_tensor.shape)

torch.Size([16, 3, 4])

## KV cache存的是什么？

传统MHA的KV Cache大小: `[2, bs, seq_len, n_kv_head * head_dim]`, 其中2代表K和V

如果我们存储MLA up后的矩阵K,V cache，那么与MHA存储无差别

那么我们可以存储kv down的矩阵：即 c_kv = w_kv_down @ h, 此时：

`[1, bs, seq_len, dc_kv]`

如果原来  2 * n_kv_head * head_dim = dim = 2 * 4096, 如果dc_kv  = 512

那么MLA KV-cache就为MHA的 1 / 16

### MLA压缩的本质是什么？

计算时间换空间

存储时刻，w_k_up

decoding时刻 即将：

`K = w_k_up @ c_kv`

`V = w_v_up @ c_kv`

压缩KV-Cache的量的意义？以VLLM来说，减少KV-Cache量。推理服务能跑更大的batch-size，从而提高inference，decoding的效率

# MLA下位置编码问题

1. 常规算法为：RoPE(W_k_up( w_k_down(h) )), 这里的RoPE没法低秩分解
2. 可以写为attention： `Rk @ Wkup @ wkdown(h)^T` `@` `[Rq @ Wqup @ wqdown(h)]^T` 
3. 上式注意到，我们所存储kv cache是wkdown(h)为 `seq x dc_kv`, 那么我们每次计算出了要up，而且还要对**所有token**增加RoPE操作
4. 有什么方式可以减少RoPE操作吗？解决方案为在h上，增加额外的参数矩阵比如：

In [14]:
class MultiHeadsLatentAttention_withRoPE(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.dim = args.dim
        # self.n_kv_heads = args.n_heads if args.n_kv_heads is None else args.n_kv_heads
        self.head_dim = args.dim // args.n_heads # 18/6 = 3
        self.dc_kv = args.dc_kv
        self.dc_q = args.dc_q

        # MLA Structure
        self.wq_down = nn.Linear(in_features=self.dim, out_features=self.dc_q, bias=False,)
        self.wq_up = nn.Linear(in_features=self.dc_q, out_features=self.dim , bias=False,)

        self.wkv_down = nn.Linear(in_features=self.dim, out_features=self.dc_kv, bias=False,)
        self.wk_up = nn.Linear(in_features=self.dc_kv, out_features=self.dim, bias=False,)
        self.wv_up = nn.Linear(in_features=self.dc_kv, out_features=self.dim, bias=False,)
        
        self.wo = nn.Linear(in_features=self.dim, out_features=self.dim,bias=False,)

        # RoPE Weight
        # K 每头一样， Q每头不一样
        self.wq_up_rope = nn.Linear(in_features=self.dc_q, out_features=self.dim , bias=False,)
        self.wk_head_rope = nn.Linear(in_features=self.dim, out_features=self.head_dim , bias=False,)

In [15]:
mla_rope = MultiHeadsLatentAttention_withRoPE(config)
print(mla_rope)

MultiHeadsLatentAttention_withRoPE(
  (wq_down): Linear(in_features=64, out_features=4, bias=False)
  (wq_up): Linear(in_features=4, out_features=64, bias=False)
  (wkv_down): Linear(in_features=64, out_features=4, bias=False)
  (wk_up): Linear(in_features=4, out_features=64, bias=False)
  (wv_up): Linear(in_features=4, out_features=64, bias=False)
  (wo): Linear(in_features=64, out_features=64, bias=False)
  (wq_up_rope): Linear(in_features=4, out_features=64, bias=False)
  (wk_head_rope): Linear(in_features=64, out_features=8, bias=False)
)

In [16]:
# 增加rope
c_q = mla_rope.wq_down(h)
xq = mla_rope.wq_up(c_q)

c_kv = mla_rope.wkv_down(h)
xk = mla_rope.wk_up(c_kv)
xv = mla_rope.wv_up(c_kv)

# print(xq.shape)
# print(xk.shape)
# print(xv.shape)

# 位置编码相关
r_q = mla_rope.wq_up(c_q) #多头
r_k = mla_rope.wk_head_rope(h) #单头
print(r_q.shape)
print(r_k.shape)

torch.Size([3, 5, 64])

torch.Size([3, 5, 8])

产生新的疑问，r_q 和 r_k 维度不同，那么做rope的dim如何处理？

In [17]:
# 简易写下
rope_matrix_q = [torch.randn(mla_rope.dim, mla_rope.dim)] * seq_len
rope_matrix_k = [torch.randn(mla_rope.head_dim, mla_rope.head_dim)] * seq_len
def apply_rope_q(x, seq_len, rope_matrix):
    for i in range(seq_len):
        x[:, i, :] = x[:, i, :] @ rope_matrix[i]
    return x

rope_q = apply_rope_q(r_q, seq_len, rope_matrix_q)
rope_k = apply_rope_q(r_k, seq_len, rope_matrix_k)

对于q每头，cat不一样的位置信息

对于k每头，cat一样的信息

In [18]:
# 与传统多头注意力无差别
xq = xq.view(bs, seq_len, mla_rope.n_heads, mla_rope.head_dim)
xk = xk.view(bs, seq_len, mla_rope.n_heads, mla_rope.head_dim)
xv = xv.view(bs, seq_len, mla_rope.n_heads, mla_rope.head_dim)


query = xq.transpose(1,2)
key = xk.transpose(1,2) # bs, n_heads, [seq_len, head_dim]
value = xv.transpose(1,2)

print(query.shape) 
print(key.shape)

## 嵌入rope
rope_q_head = rope_q.view(bs, seq_len, mla_rope.n_heads, mla_rope.head_dim)
rope_q_head = rope_q_head.transpose(1,2)

rope_k_head = rope_k.unsqueeze(dim = 1).repeat( repeats = [1, mla_rope.n_heads, 1, 1])
print(rope_q_head.shape)
print(rope_k_head.shape)


# cat 操作
query_cat = torch.cat((query, rope_q_head), dim = -1)
key_cat = torch.cat((query, rope_k_head), dim = -1)

torch.Size([3, 8, 5, 8])

torch.Size([3, 8, 5, 8])

torch.Size([3, 8, 5, 8])

torch.Size([3, 8, 5, 8])

In [19]:
### 常规attention
# scores = query @ key.transpose(2,3) # keys: bs, n_heads, [head_dim, seq_len]
scores = query_cat @ key_cat.transpose(2,3) # keys: bs, n_heads, [2head_dim, seq_len]
scores = F.softmax(scores.float(), dim=-1).type_as(xq)
output = scores @ value 
output = output.transpose(1, 2).contiguous().view(bs, seq_len, -1)
output = mla.wo(output)
print(output.shape)

torch.Size([3, 5, 64])

## 增加问题

1. 为什么位置编码要分离
2. q和k的位置编码维度不一样，那么rope的维度是否一样
3. v_up 如何被 wo 吸收
5. 写出带kv_cache版本的MLA，及decoding代码
6. MLA训练的显存计算
7. MLA并行参数分配、张量并行细节和通信